# 07 · Modelos densos y diversidad (Fase 14)

**Pregunta.** Sin posibilidad de usar embeddings (restricción de la competencia), ¿aportan algo
los modelos no lineales sobre representaciones densas?

**Nota de entorno (decisión D-007).** Al iniciar esta fase la GPU dedicada estaba **ocupada por
una carga ajena** (uso 42% sostenido, sin PIDs de cómputo propios). El protocolo del proyecto
ordena **no iniciar trabajo en GPU y devolver el control al usuario**; por eso esta fase se ejecutó
en **CPU**. El evento quedó registrado en `experiments/gpu_events.log`.

## 0. Configuración

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config as cfg
from src.config import set_seed

set_seed()
pd.set_option("display.width", 175)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 70)

def load_json(name):
    return json.loads((cfg.EXPERIMENTS / name).read_text(encoding="utf-8"))

clasicos = load_json("modelos_clasicos_resultados.json")
densos = load_json("e019_modelos_densos.json")
ensamble_mlp = load_json("e020_mlp_en_ensamble.json")
errores = load_json("fase17_error_analysis.json")
barrido_c = load_json("barrido_c_resultados.json")
print("artefactos cargados")

artefactos cargados


## 1. Resultados de los MLP

In [2]:
tabla = pd.DataFrame([
    {"id": "E-019a", "modelo": "MLP sobre 35 features densas",
     "macro_f1": round(densos["E-019a"]["macro_f1_mean"], 4),
     "std": round(densos["E-019a"]["macro_f1_std"], 4),
     "accuracy": round(densos["E-019a"]["accuracy_mean"], 4)},
    {"id": "E-019b", "modelo": "MLP sobre 35 densas + SVD(64) del espacio de caracteres",
     "macro_f1": round(densos["E-019b"]["macro_f1_mean"], 4),
     "std": round(densos["E-019b"]["macro_f1_std"], 4),
     "accuracy": round(densos["E-019b"]["accuracy_mean"], 4)},
])
tabla.to_csv(cfg.TABLES / "modelos_densos_comparativa.csv", index=False)
print("entorno de ejecución:", densos["entorno"])
tabla

entorno de ejecución: CPU (GPU ocupada por carga ajena; protocolo D-007)


,id,modelo,macro_f1,std,accuracy
0,E-019a,MLP sobre 35 features densas,0.3381,0.004,0.3492
1,E-019b,MLP sobre 35 densas + SVD(64) del espacio de caracteres,0.4071,0.006,0.4107


**OBSERVACIÓN.** Ambos MLP quedan muy por debajo de los modelos lineales dispersos: 0,3381
(densas) y 0,4071 (densas + SVD) frente a 0,5658 del mejor individual.

**INTERPRETACIÓN.** La reducción SVD conserva apenas el **7,8% de la varianza** en 64 componentes:
el espacio de 314k n-gramas de carácter es demasiado disperso para comprimirlo linealmente sin
perder señal. Es el argumento cuantitativo de por qué la vía densa pierde en este corpus.

**CONSECUENCIA.** El deep learning **no** se adopta como modelo principal.

## 2. El MLP como miembro diverso del ensamble

In [3]:
comparativa = pd.DataFrame([
    {"ensamble": "9 fuentes + densas (E-018, referencia)",
     "macro_f1": round(ensamble_mlp["e018_referencia"]["macro_f1"], 4),
     "std": round(ensamble_mlp["e018_referencia"]["std"], 4)},
    {"ensamble": "9 fuentes + MLP densas+SVD + densas (E-020)",
     "macro_f1": round(ensamble_mlp["e020_con_mlp"]["macro_f1"], 4),
     "std": round(ensamble_mlp["e020_con_mlp"]["std"], 4)},
])
comparativa["delta"] = comparativa["macro_f1"].diff().round(4)
comparativa.to_csv(cfg.TABLES / "mlp_en_ensamble.csv", index=False)
print("aporte del MLP al ensamble:", round(ensamble_mlp["aporte"], 4))
comparativa

aporte del MLP al ensamble: 0.0066


,ensamble,macro_f1,std,delta
0,"9 fuentes + densas (E-018, referencia)",0.5784,0.0044,NaN
1,9 fuentes + MLP densas+SVD + densas (E-020),0.5851,0.0042,0.0067


**HALLAZGO CENTRAL DE LA FASE.** El MLP rinde **0,4071 en solitario** (16 puntos por debajo del
mejor lineal) pero aporta **+0,0066 al ensamble**, el mayor aporte individual desde el canal de
menciones.

**INTERPRETACIÓN.** Su valor no está en la exactitud sino en la **decorrelación de sus errores**:
al equivocarse en casos distintos, el meta-modelo aprende a usarlo donde los modelos lineales
fallan. Es la lección transferible del proyecto sobre modelos no lineales sin embeddings.

**DECISIÓN (D-015).** El MLP entra al ensamble como miembro diverso; la submission v06 lo
incorpora y obtiene 0,5851 de CV y **0,60273** en el leaderboard.